In [1]:
import jax
import pgx
from pgx.experimental import act_randomly
from twentyfourtyeight import *
import jax.numpy as jnp

In [2]:
print(f"{pgx.__version__=}")

env = pgx.make("2048")

init = jax.jit(jax.vmap(env.init))  # vectorize and JIT-compile
step = jax.jit(jax.vmap(env.step))
act_randomly = jax.jit(act_randomly)

batch_size = 32

# prepare PRNGKeys
key = jax.random.key(42)
key, subkey = jax.random.split(key)
keys = jax.random.split(subkey, batch_size)

# state = init(keys)  # vectorized states
# while not (state.terminated | state.truncated).all():
#     key, subkey = jax.random.split(key)
#     action = act_randomly(subkey, state.legal_action_mask)
#     print(action.shape)
#     # actions = jax.random()
#     # actions = jnp.zeros((32, 4))
#     # actions = jnp.where(state.legal_action_mask, actions, -jnp.inf)
#     # actions = jax.nn.softmax(actions)
#     # print(actions)
#     key, subkey = jax.random.split(key)
#     keys = jax.random.split(subkey, batch_size)
#     state = step(state, action, keys)  # state.reward (2,)

pgx.__version__='2.5.0'


In [6]:
env = pgx.make("2048")

init = jax.jit(jax.vmap(env.init))  # vectorize and JIT-compile
step = jax.jit(jax.vmap(env.step))

key = jax.random.key(42)
key, subkey = jax.random.split(key)
keys = jax.random.split(subkey, batch_size)

state = init(keys)  # vectorized states
key, subkey = jax.random.split(key)
model = MLP(496, 4, subkey)
obs_wrapper = ToInt(FlattenObservation())
policy = model.policy_fn
current_obs, traj = collect_trajectory(step, state, state.observation, model.policy_fn, key, 16, obs_wrapper)

In [10]:
traj.observations.shape

(16, 32, 4, 4, 31)